#Install

In [0]:
%pip install requests beautifulsoup4

#Import

In [0]:
import requests
import hashlib
from bs4 import BeautifulSoup
import urllib.parse
from datetime import datetime
from pyspark.sql.types import StructType, StructField, StringType, LongType, TimestampType

#Configure

In [0]:
BLS_URL = 'https://download.bls.gov/pub/time.series/pr/'
S3_BUCKET = 'rearc-quest-107628756615-us-east-2-an'
S3_PREFIX = 'bls-data/pr/'
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
}

# Create Delta manifest table
spark.sql("""
    CREATE TABLE IF NOT EXISTS quest.bls_pr_manifest (
        filename STRING NOT NULL,
        checksum STRING NOT NULL,
        size LONG NOT NULL,
        last_synced TIMESTAMP NOT NULL
    )
    USING DELTA
""")

#Define

In [0]:
def scrape_bls_pr():
    """Scrape BLS PR directory and compute checksums for all files."""
    response = requests.get(BLS_URL, timeout=30, headers=HEADERS)
    soup = BeautifulSoup(response.content, 'html.parser')
    
    files = []
    
    for link in soup.find_all('a'):
        href = link.get('href', '')
        filename = link.get_text(strip=True)
        
        # Skip parent dir and directories
        if href in ['..', '../', './', ''] or href.endswith('/'):
            continue
        
        # Build full URL
        if href.startswith('/'):
            full_url = f"https://download.bls.gov{href}"
        else:
            full_url = urllib.parse.urljoin(BLS_URL, href)
        
        try:
            content = requests.get(full_url, timeout=30, headers=HEADERS).content
            checksum = hashlib.md5(content).hexdigest()
            
            files.append({
                'filename': filename,
                'checksum': checksum,
                'size': len(content),
                '_content': content  # Temporary storage for upload
            })
        except Exception as e:
            print(f"✗ {filename}: {str(e)}")
    
    return files

#Process

##1. Scrape BLS PR directory

In [0]:
current_files = scrape_bls_pr()
print(f"✓ Found and checksummed {len(current_files)} files\n")

##2. Compare with manifest

In [0]:
# Load old manifest from Delta
try:
    old_manifest = {row.filename: row.checksum 
                    for row in spark.sql("SELECT filename, checksum FROM quest.bls_pr_manifest").collect()}
except:
    old_manifest = {}

# Categorize files
to_upload = [f for f in current_files if f['filename'] not in old_manifest]
to_reupload = [f for f in current_files if f['filename'] in old_manifest 
               and old_manifest[f['filename']] != f['checksum']]
unchanged = [f for f in current_files if f['filename'] in old_manifest 
             and old_manifest[f['filename']] == f['checksum']]

print(f"New files:       {len(to_upload)}")
print(f"Updated files:   {len(to_reupload)}")
print(f"Unchanged files: {len(unchanged)}")

if to_upload:
    print(f"\nNew:")
    for f in to_upload:
        print(f"  + {f['filename']}")

if to_reupload:
    print(f"\nUpdated:")
    for f in to_reupload:
        print(f"  ~ {f['filename']}")

##3. Upload to s3

In [0]:
print("\n[3/4] Uploading to S3...")

upload_count = 0
for f in to_upload + to_reupload:
    s3_path = f"s3://{S3_BUCKET}/{S3_PREFIX}{f['filename']}"
    try:
        # dbutils.fs.put writes file content to S3
        dbutils.fs.put(s3_path, f['_content'].decode('utf-8'), overwrite=True)
        print(f"  ✓ {f['filename']:30} -> {s3_path}")
        upload_count += 1
    except Exception as e:
        print(f"  ✗ {f['filename']}: {str(e)}")

print(f"\n✓ Uploaded {upload_count} files")

##4. Handle deleted files

In [0]:
# Find files in manifest that are no longer on BLS
deleted_files = [f for f in old_manifest if f not in {x['filename'] for x in current_files}]

if deleted_files:
    print(f"Deleted files: {len(deleted_files)}")
    for filename in deleted_files:
        s3_path = f"s3://{S3_BUCKET}/{S3_PREFIX}{filename}"
        try:
            dbutils.fs.rm(s3_path)
            print(f"  ✓ {filename:30} -> deleted from S3")
        except Exception as e:
            print(f"  ✗ {filename}: {str(e)}")
    
    # Remove deleted files from manifest
    spark.sql(f"""
        DELETE FROM quest.bls_pr_manifest
        WHERE filename IN ({','.join([f"'{f}'" for f in deleted_files])})
    """)
    print(f"✓ Removed {len(deleted_files)} files from manifest")
else:
    print("No deleted files")

##5. Update Delta manifest

In [0]:
# Create DataFrame of current files (exclude _content)
schema = StructType([
    StructField("filename", StringType(), False),
    StructField("checksum", StringType(), False),
    StructField("size", LongType(), False),
    StructField("last_synced", TimestampType(), False),
])

data = [(f['filename'], f['checksum'], f['size'], datetime.utcnow()) 
        for f in to_upload + to_reupload]

df = spark.createDataFrame(data, schema=schema)
df.createOrReplaceTempView("temp_updates")

# Merge (upsert) into manifest
spark.sql("""
    MERGE INTO quest.bls_pr_manifest AS target
    USING temp_updates AS source
    ON target.filename = source.filename
    WHEN MATCHED THEN
        UPDATE SET 
            checksum = source.checksum,
            size = source.size,
            last_synced = source.last_synced
    WHEN NOT MATCHED THEN
        INSERT *
""")

print("\n" + "="*80)
print("SYNC COMPLETE")
print("="*80)


#Show current manifest state

In [0]:
display(spark.sql("""
    SELECT 
        filename,
        round(size / 1024.0, 1) as size_kb,
        checksum,
        last_synced
    FROM quest.bls_pr_manifest
    ORDER BY last_synced DESC
"""))